# Проект. Анализ игроковй индустрии
 29.11.2025


## Цели и задачи проекта

Изучить историю продаж игр в начале XXI века, в периоды с 2000 по 2013 год включительно. Подготовить данные и разделить игры на категории на основе оценок критиков и пользователей.

### Описание данных

- `Name` — название игры.
- `Platform` — название платформы.
- `Year of Release` — год выпуска игры.
- `Genre` — жанр игры.
- `NA sales` — продажи в Северной Америке (в миллионах проданных копий).
- `EU sales` — продажи в Европе (в миллионах проданных копий).
- `JP sales` — продажи в Японии (в миллионах проданных копий).
- `Other sales` — продажи в других странах (в миллионах проданных копий).
- `Critic Score` — оценка критиков (от 0 до 100).
- `User Score` — оценка пользователей (от 0 до 10).
- `Rating` — рейтинг организации ESRB (англ. Entertainment Software Rating Board). Эта ассоциация определяет рейтинг компьютерных игр и присваивает им подходящую возрастную категорию.

### Содержимое проекта

- [Загрузка и знакомство с данными](#a0).
- [Проверка ошибок в данных и их предобработка](#a1)
- [Наличие пропусков в данных](#a2)
- [Фильтрация данных](#a3)
- [Категоризация данных](#a4)
- [Итоговый вывод](#a5)

<a id='a0'></a>

##  Загрузка данных и знакомство с ними

- Загрузите необходимые библиотеки Python и данные датасета `/datasets/new_games.csv`.


Загрузим необходимые библиотеки для анализа данных и данные из датасета `/datasets/new_games.csv`. Отберем данные `с 2000 по 2013`. Затем выведем основную информацию о данных с помощью метода `info()` и первые строки датафрейма.

In [1]:
# ипортируем библиатеку для работы в Pandas
import pandas as pd

# ипортируем библиатеку NumPy, понадобится для работы с пропусками  
import numpy as np


In [2]:
# Указываем путь к файлу
data = pd.read_csv('https://code.s3.yandex.net/datasets/new_games.csv')

row_count = data.shape[0]

In [3]:
# Выводим информацию о датафрейме
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16954 non-null  object 
 1   Platform         16956 non-null  object 
 2   Year of Release  16681 non-null  float64
 3   Genre            16954 non-null  object 
 4   NA sales         16956 non-null  float64
 5   EU sales         16956 non-null  object 
 6   JP sales         16956 non-null  object 
 7   Other sales      16956 non-null  float64
 8   Critic Score     8242 non-null   float64
 9   User Score       10152 non-null  object 
 10  Rating           10085 non-null  object 
dtypes: float64(4), object(7)
memory usage: 1.4+ MB


In [4]:
# Выводим первые строки датафрейма на экран
data.head()

,Name,Platform,Year of Release,Genre,NA sales,EU sales,JP sales,Other sales,Critic Score,User Score,Rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN


Предварительный вывод по датасету `hotel_dataset.csv`:

- содержит 11 столбцов и 16956 строк, в которых представлена информация о выпущенных играх с 2000 по 2013 г;
- почти в каждом строке имеются пропуски;
- в основном используется тип данных `object` (тип данных для хранения любых объектов Python) и `float64`. В дальнейшем изменим тип данных `float64` и приведем для числовых значений к типу`float64` и `int` для, как например для `NA sales`,`EU sales`, `JP sales` (объемы продаж в разных регионах);
- так как в большинсте числовых данных был выбран автоматически тип данных `object`, то это может указывать, что в данных могут быть ошибке в виде текста вместо цифр;
- `Year of Release` использует некорректный тип данных для вывода информации о годе издания игры;
- часть информации об оценке критиков `Critic Score` и пользователей `User Score` пропущена (указано `NaN`), что в дальнейшем может повлиять на результаты работы;
- информация о рейтинге ESRB `Rating` игры также не полная и содержит пропуски.

<a id='a1'></a>

---

## Проверка ошибок в данных и их предобработка


### Названия, или метки, столбцов датафрейма

- Выведите на экран названия всех столбцов датафрейма и проверьте их стиль написания.
- Приведите все столбцы к стилю snake case. Названия должны быть в нижнем регистре, а вместо пробелов — подчёркивания.

In [5]:
display(data.columns)

Index(['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales',
       'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating'],
      dtype='object')

In [6]:
#смотрим, что получилось 
data.columns = data.columns.str.lower().str.replace(' ', '_')
display(data.columns)

Index(['name', 'platform', 'year_of_release', 'genre', 'na_sales', 'eu_sales',
       'jp_sales', 'other_sales', 'critic_score', 'user_score', 'rating'],
      dtype='object')

In [7]:
# Выводим первые строки датафрейма на экран
data.head()

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN


### Типы данных

- Если встречаются некорректные типы данных, предположите их причины.
- При необходимости проведите преобразование типов данных. Помните, что столбцы с числовыми данными и пропусками нельзя преобразовать к типу `int64`. Сначала вам понадобится обработать пропуски, а затем преобразовать типы данных.

#### Первичный анализ данных и поиск возможных в них ошибок, оптимизация типов данных

1. `platform`, тип object. Тип данных оптимален. Проверка `platform` на наличие ошибок. Сделаем проверку перед `name`, так как эта колонка понадобится в дальнйшем. 

In [8]:
unique_values = data['platform'].unique()
sorted_values = sorted(unique_values)
print("Отсортированные уникальные значения в столбце 'platform':", sorted_values)

Отсортированные уникальные значения в столбце 'platform': ['2600', '3DO', '3DS', 'DC', 'DS', 'GB', 'GBA', 'GC', 'GEN', 'GG', 'N64', 'NES', 'NG', 'PC', 'PCFX', 'PS', 'PS2', 'PS3', 'PS4', 'PSP', 'PSV', 'SAT', 'SCD', 'SNES', 'TG16', 'WS', 'Wii', 'WiiU', 'X360', 'XB', 'XOne']


**Вывод: ошибки отсутствуют, дуликатов нет**

2. `year_of_release` тип данных object. Если поменять тип данных на `int16`, то выходит ошибка `Cannot convert non-finite values (NA or inf) to integer`. Решим проблему и помчитаем количество пропусков в данных:

In [9]:
gaps_in_column = data['year_of_release'].isna().sum()
print("Всего пропусков:", gaps_in_column)

Всего пропусков: 275


In [10]:
# numpy as np - перенес начало работы
# Заменяем NaN и inf на 0
data['year_of_release'].fillna(-1, inplace=True)

# Теперь преобразуем в int16
data['year_of_release'] = data['year_of_release'].astype('int16')

C:\Users\torun\AppData\Local\Temp\ipykernel_16188\732653364.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['year_of_release'].fillna(-1, inplace=True)


3. `name` тип object. Тип данных оптимален. Проверка `name`: есть ли дубли. Проверим гипотезу, что некоторые игры могли быть выпущены в разные года и могут иметь разные жанры и поэтому могут повторяться.

In [11]:
# Преобразование 'year_of_release' в текст, чтобы исключить символы, которые не относятся к году.
data['year_of_release'] = data['year_of_release'].astype(str)
# Соединяем столбцы 'name' и 'platform' и 'year_of_release', преобразуя их в верхний регистр
data['combined'] = data['name'].str.upper() + ' ' + data['platform'].str.upper()+' ' + data['year_of_release'].str.upper()
#Отсортируем по колонке combined для удобства анализа итоговой таблицы
data = data.sort_values(by='combined')
# Проверяем наличие дубликатов в новом столбце
duplicates = data['combined'].duplicated(keep =False)
duplicates_count = duplicates.sum()
duplicates_rows = data.loc[duplicates]

print(f"Количество дубликатов в объединённом столбце: {duplicates_count/2}")
print(f"Сформируем таблицу, где присутствуют дубликаты по столбцу name")
#вернем обратно для 'year_of_release' тип данных`int16`
data['year_of_release'] = data['year_of_release'].astype('int16')
display(duplicates_rows)

Количество дубликатов в объединённом столбце: 243.0
Сформируем таблицу, где присутствуют дубликаты по столбцу name


,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating,combined
15192,Beyblade Burst,3DS,2016,Role-Playing,0.00,0.0,0.03,0.00,NaN,NaN,NaN,BEYBLADE BURST 3DS 2016
15191,Beyblade Burst,3DS,2016,Role-Playing,0.00,0.0,0.03,0.00,NaN,NaN,NaN,BEYBLADE BURST 3DS 2016
15302,11eyes: CrossOver,X360,2009,Adventure,0.00,0.0,0.02,0.00,NaN,NaN,NaN,11EYES: CROSSOVER X360 2009
15301,11eyes: CrossOver,X360,2009,Adventure,0.00,0.0,0.02,0.00,NaN,NaN,NaN,11EYES: CROSSOVER X360 2009
4860,18 Wheeler: American Pro Trucker,PS2,2001,RACING,0.20,0.15,0.0,0.05,61.0,5.7,E,18 WHEELER: AMERICAN PRO TRUCKER PS2 2001
...,...,...,...,...,...,...,...,...,...,...,...,...
6695,Zoo Resort 3D,3DS,2011,Simulation,0.11,0.09,0.03,0.02,NaN,tbd,E,ZOO RESORT 3D 3DS 2011
8156,Zumba Fitness Rush,X360,2012,SPORTS,0.00,0.16,0.0,0.02,73.0,6.2,E10+,ZUMBA FITNESS RUSH X360 2012
8157,Zumba Fitness Rush,X360,2012,Sports,0.00,0.16,0.0,0.02,73.0,6.2,E10+,ZUMBA FITNESS RUSH X360 2012
661,NaN,GEN,1993,NaN,1.78,0.53,0.0,0.08,NaN,NaN,NaN,NaN


**Сделаем вывод**, что эти данные задублированы, в чем можно убедиться, если взглянуть на сформированную таблицу. Все данные идентичны. Также стоит отметить, что у части дубликатов жанр игры написан в разном регистре. Поправим в дальнйшем жанр, чтобы информация была представлена одинаковым способов.

**Удалим найденне дубликаты**

In [12]:
# удалим 200 строчек с дубликатом данных по колонке name+platform+year_of_release
duplicates_count = data[data.duplicated(subset=['combined'])].shape[0]
print(f"Количество удаленных строк с дубликатами: {duplicates_count}")
data = data.drop_duplicates(subset=['combined'], keep='first')

Количество удаленных строк с дубликатами: 243


4. `genre` тип данных object, то оптимально. Приведем колонку `genre` к одному регистру.

In [13]:
data['genre'] = data['genre'].str.lower()

5. `na_sales`, `eu_sales`, `jp_sales`, `other_sales` тип данных object. Оптимальнее использовать тип float64. Проверим, есть ли в ошибки, учитывая, что пропусков нет. Используем статистические методы для проверки. Если попытаться применить сложение к данным, то выйдет ошибка, так как часть данных в тектосовом формате. Переведем к типу данных float64.

In [14]:
# Заменим строки 'unknown' на NaN и преобразуем float64
data['na_sales'] = pd.to_numeric(data['na_sales'], errors='coerce')
data['eu_sales'] = pd.to_numeric(data['eu_sales'], errors='coerce')
data['jp_sales'] = pd.to_numeric(data['jp_sales'], errors='coerce')


In [15]:
# проверим, получилось ли решить проблему с текстовыми данными
result = data[['na_sales', 'eu_sales', 'jp_sales', 'other_sales']].agg([
    'sum',          # сумма
    'mean',         # среднее
    'min',          # минимум
    'max',          # максимум
    'median'        # медиана
])

print(result)

           na_sales     eu_sales    jp_sales  other_sales
sum     4402.350000  2420.810000  1297.28000   791.320000
mean       0.263409     0.144898     0.07764     0.047348
min        0.000000     0.000000     0.00000     0.000000
max       41.360000    28.960000    10.22000    10.570000
median     0.080000     0.020000     0.00000     0.010000


**Промежуточный вывод**. Тип данных изменен. Ошибок не выявлено.

In [16]:
# Замена строки 'unknown' на NaN
data['eu_sales'] = data['eu_sales'].replace('unknown', np.nan)
data['jp_sales'] = data['jp_sales'].replace('unknown', np.nan)
# Теперь преобразуем в float64
data['na_sales'] = data['na_sales'].astype('float64')
data['eu_sales'] = data['eu_sales'].astype('float64')
data['jp_sales'] = data['jp_sales'].astype('float64')

Повторим анализ данных

In [17]:
result = data[['na_sales', 'eu_sales', 'jp_sales', 'other_sales']].agg([
    'sum',          # сумма
    'mean',         # среднее
    'min',          # минимум
    'max',          # максимум
    'median'        # медиана
])

print(result)

           na_sales     eu_sales    jp_sales  other_sales
sum     4402.350000  2420.810000  1297.28000   791.320000
mean       0.263409     0.144898     0.07764     0.047348
min        0.000000     0.000000     0.00000     0.000000
max       41.360000    28.960000    10.22000    10.570000
median     0.080000     0.020000     0.00000     0.010000


**Вывод:** ошибки устранены.

6. `critic_score`, `user_score` тип object. Приведем к типу float64. Используем статистические методы для проверки. **При визуальном анализе данных, выявлено, что ошибки в данных: значения 'tbd'. Исправим это и приведем к типу float64.**

In [18]:
# Сначала заменим ошибку в данных и потом безопасно преобразуем в float64
data['critic_score'] = data['critic_score'].replace('tbd', np.nan)
data['user_score'] = data['user_score'].replace('tbd', np.nan)
# поменяем тип данных на float64
data['critic_score'] = data['critic_score'].astype('float64')
data['user_score'] = data['user_score'].astype('float64')

7. `rating` имеет тип object. Содержит как текст так и цифры. Приведем все к к верхнему регистру, что считаем достаточным. Для задач анализа эта информация не используется. Поэтому считаем, что возможные ошибки в данной колонке не повлияют на итоги анализа.

In [19]:
data['rating'] = data['rating'].str.upper()

**Выведем информацию о таблице после проверки на ошибки и их исправления, удалив колонку `combined`**

In [20]:
data = data.drop(columns=['combined'])
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16713 entries, 15192 to 661
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16712 non-null  object 
 1   platform         16713 non-null  object 
 2   year_of_release  16713 non-null  int16  
 3   genre            16712 non-null  object 
 4   na_sales         16713 non-null  float64
 5   eu_sales         16707 non-null  float64
 6   jp_sales         16709 non-null  float64
 7   other_sales      16713 non-null  float64
 8   critic_score     8136 non-null   float64
 9   user_score       7589 non-null   float64
 10  rating           9948 non-null   object 
dtypes: float64(6), int16(1), object(4)
memory usage: 1.4+ MB


### Наличие пропусков в данных

В этом разделе посмотрим на количество пропущенных значений по столбцам датафрейма (учтем, что на прошлом шаге часть пропущенных данных или ошибок были исправлены или заменены)

In [21]:
# Подсчитываем количество пропущенных значений для каждого столбца
missing_count = data.isna().sum()

# Вычисляем процент пропущенных значений
missing_percentage = (data.isna().sum() / len(data)) * 100

# Создаём новый датафрейм из полученных данных
missing_data = pd.DataFrame({
    'Missing Count': missing_count,
    'Missing Percentage': missing_percentage
})

# Отображаем результат
display(missing_data)

,Missing Count,Missing Percentage
name,1,0.005983
platform,0,0.000000
year_of_release,0,0.000000
genre,1,0.005983
na_sales,0,0.000000
eu_sales,6,0.035900
jp_sales,4,0.023933
other_sales,0,0.000000
critic_score,8577,51.319332
user_score,9124,54.592234


**Всего 16713 строк.**
В данных наблюдаются пропуски в следующих столбцах:
- `name`: в 1 строке (0.0059% данных) отсутствует информация о названии игры. Вероятно это ошибка свзана с занесением информации в таблицу или с выгрузкой. Удалим, так как качество данных от этого не пострадает.
- `genre`: в 1 строке (0.0059% данных) отсутствует информация о жанре игры. Вероятно это ошибка свзана с занесением информации в таблицу или с выгрузкой. Оставим это значение без изменений, так как на цели и задачи данной работы это не влияет.
- `eu_sales`: в 6 строках (0.036% данных) отсутствует информация о продаже в Европе. Вероятно продаж не было. Но чтобы не искажать данные, удалим их.
- `jp_sales`: в 4 строках (0.024% данных) отсутствует информация о продаже в Японии. Вероятно продаж не было. Но чтобы не искажать данные, удалим их.
- `critic_score`: в 8577 строках (51.32% данных) отсутствует информация об оценке критиков. Пропуски в этом столбце могут затруднить анализ игр по оценкам пользователей и экспертов.
- `user_score`: в 9124 строках (54.59% данных) отсутствует информация об оценке пользователей . Пропуски в этом столбце могут затруднить анализ игр по оценкам пользователей и экспертов.
- `rating`: в 6765 строках (40.48% данных) отсутствует информация о рейтинге ESRB.


In [22]:
#удалим пропуски в `eu_sales` и `jp_sales`
data = data.dropna(subset=['eu_sales', 'jp_sales'])


**Вывод:** множество пропущенных данных по `critic_score` и `user_score` могут быть связаны отсутствием данных в открытых источниках, слабым интересом игроков и критиков к определенным играм. Возможно в будущем стоит использовать большее число источников информации при поиске данных. Было бы интересно сравнить как количество продаж игр связано с оценами критиков и пользователей.

**Посчитаем количество удалённых строк в абсолютном и относительном значениях**

In [23]:
print(f"Количество строк до удаления: {row_count}")
row_count_1= row_count -data.shape[0]
print(f"количество удалённых строк: {row_count_1} или {row_count_1 / row_count * 100:.2f}%")
print(f"Количество строк после удаления: {data.shape[0]}")


Количество строк до удаления: 16956
количество удалённых строк: 253 или 1.49%
Количество строк после удаления: 16703


### Явные и неявные дубликаты в данных

Согласно задаче, явные и неявные дубликаты были выявлены на предыдущиъ этапах проекта. На текущем этапе дубликаты устранены. 

<a id='a3'></a>

---

## Фильтрация данных

Изучим историю продаж игр в начале XXI века, в периоды с 2000 по 2013 год включительно. Отберем данные по этому показателю. Сохраним новый срез данных в отдельном датафрейме, `df_actual`.

In [24]:
df_actual = data.loc[(data['year_of_release'] >= 2000) & (data['year_of_release'] <= 2013)].copy()


<a id='a4'></a>

---

## Категоризация данных
    
1. Разделим все игры по оценкам пользователей и выделим такие категории: 
- высокая оценка (от 8 до 10 включительно)
- средняя оценка (от 3 до 8, не включая правую границу интервала) 
- низкая оценка (от 0 до 3, не включая правую границу интервала)
2. Разделим все игры по оценкам критиков и выделим такие категории: 
- высокая оценка (от 80 до 100 включительно)
- средняя оценка (от 30 до 80, не включая правую границу интервала) 
- низкая оценка (от 0 до 30, не включая правую границу интервала)

In [25]:
def categorize_critic_score(value):
    if value < 30:
        return 'низкая оценка критиков'
    elif 30 <= value < 80:
        return 'средняя оценка критиков'
    elif 80 <= value <= 100:
        return 'высокая оценка критиков'
    else:
        return 'Нет данных'

# создадим функцию для категорий по user_score
def categorize_user_score(value):
    if value < 3:
        return 'низкая оценка игроков'
    elif 3 <= value < 8:
        return 'средняя оценка игроков'
    elif 8 <= value <= 10:
        return 'высокая оценка игроков'
    else:
        return 'Нет данных'
    
# Применение функции к данным
df_actual['category_critic_score'] = data['critic_score'].apply(categorize_critic_score)
df_actual['category_user_score'] = data['user_score'].apply(categorize_user_score)

In [26]:
df_actual.head()

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating,category_critic_score,category_user_score
3394,Frozen: Olaf's Quest,3DS,2013,platform,0.27,0.27,0.00,0.05,NaN,NaN,NaN,Нет данных,Нет данных
3906,Frozen: Olaf's Quest,DS,2013,platform,0.21,0.26,0.00,0.04,NaN,NaN,NaN,Нет данных,Нет данных
2478,Tales of Xillia 2,PS3,2012,role-playing,0.20,0.12,0.45,0.07,71.0,7.9,T,средняя оценка критиков,средняя оценка игроков
8460,.hack//G.U. Vol.1//Rebirth,PS2,2006,role-playing,0.00,0.00,0.17,0.00,NaN,NaN,NaN,Нет данных,Нет данных
8719,.hack//G.U. Vol.2//Reminisce (jp sales),PS2,2006,role-playing,0.00,0.00,0.16,0.00,NaN,NaN,NaN,Нет данных,Нет данных


In [27]:
df_actual.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12771 entries, 3394 to 9260
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   name                   12771 non-null  object 
 1   platform               12771 non-null  object 
 2   year_of_release        12771 non-null  int16  
 3   genre                  12771 non-null  object 
 4   na_sales               12771 non-null  float64
 5   eu_sales               12771 non-null  float64
 6   jp_sales               12771 non-null  float64
 7   other_sales            12771 non-null  float64
 8   critic_score           7161 non-null   float64
 9   user_score             6475 non-null   float64
 10  rating                 8714 non-null   object 
 11  category_critic_score  12771 non-null  object 
 12  category_user_score    12771 non-null  object 
dtypes: float64(6), int16(1), object(6)
memory usage: 1.3+ MB


**После категоризации данных проверим результат: сгруппируем данные по выделенным категориям и посчитаем количество игр в каждой категории.**

In [28]:
category_counts = df_actual.groupby('category_critic_score').size().reset_index(name='Количество игр')
print(category_counts)

     category_critic_score  Количество игр
0               Нет данных            5610
1  высокая оценка критиков            1685
2   низкая оценка критиков              55
3  средняя оценка критиков            5421


**Выделим топ-7 платформ по количеству игр**

In [29]:
platform_counts = data.groupby('platform')['name'].count().sort_values(ascending=False).reset_index()
platform_counts.index = platform_counts.index + 1
print('Cамые популярные платформы:')
print(platform_counts.head(7))

Cамые популярные платформы:
  platform  name
1      PS2  2160
2       DS  2148
3      PS3  1330
4      Wii  1320
5     X360  1259
6      PSP  1208
7       PS  1197


<a id='a5'></a>

---

## Итоговый вывод


Были загружены данные `datasets/new_games.csv`и отфильтрованы по годам с 2000 по 2013. Данные содержали 11 столбцов и 12780 строк, в которых представлена исторические данные из открытых источников о продажах игр, сделанных в разных жанрах и выпущенных на разных платформах, а также пользовательские и экспертные оценки игр. При первичном знакомстве с данными и их предобработке получили такие результаты:
- Все столбцы приведены к стилю snake case
- При анализе столбцов `name`, `platform`, `genre` было выявлено и удалено 243 дубликатов или 1.43%.
- В пяти столбцах (`eu_sales`, `jp_sales`, `critic_score`, `user_score`, `rating`) были обнаружены пропущенные значения. Для `eu_sales`, `jp_sales` пропуски были заменены на 0. Для `critic_score`, `user_score`, `rating` пропуски остались без изменений. 
- Для `genre` был изменен регистр на нижний, так как были замечены одни и те же данные написанные в верхнем или нижнем регистрах.
- Для оптимизации работы с данными в датафрейме были произведены следующие изменения типов данных:
    - `Year of Release`: тип данных изменён с `float64` на `int16`.
    - `na_sales`, `eu_sales`, `jp_sales`, `critic_score`, `user_score` тип данных был приведен к `object`.

- Для дополнительной работы с данными были созданы дополнительные столбцы:
    - `category_critic_score`: показывает категорию игры на основе оценок критиков;
    - `category_user_score`: показывает категорию игры на основе оценок игроков;
 
Получилось очистить данные от ошибок и провести знакомство с полученными данными. Но большое количество пропусков не дает возможность достоверно провести оценку полученных данных и создать достоверный рейтинг игр на основе оценок криктиков и игроков. Возможно в будущем стоит использовать большее число источников информации для поиске данных.